# ISG vectors compared by cosine similarity

Each text of a pair is its integrated syntactic graph (ISG) vector, built once by the extraction
notebook, and the pair is "same author" when the cosine similarity of its two vectors reaches a
threshold. Nothing is tuned or trained - the threshold is selected on the validation pairs from every
similarity observed there and applied once to the test pairs.

A finished run is not repeated; a killed run starts again from the top (the similarities take minutes).

In [ ]:
SMOKE = False  # True - the first rows of every data set, the smoke vectors
RUN_TAG = ""  # label in the file names of a repeated run

Standard libraries

In [ ]:
import hashlib
import logging
import re
import tempfile
import time
from itertools import islice
from pathlib import Path

Data and plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix

The shared modules of the thesis

In [ ]:
from avcore import data, evaluation, features, graph, progress, runs, tracking

# Configuration

The decision rule and the developer of the run

In [ ]:
CLASSIFIER = "cosine_similarity"
DEVELOPER_INITIALS = "JP"

What the run is, in the words of the result tables

In [ ]:
RUN_IDENTITY = {
    "family": "graph",
    "model": "none",
    "representation": "isg",
    "combination": "none",
    "classifier": CLASSIFIER,
}

The smoke run - the extraction notebook builds its smoke folder from the same rows

In [ ]:
SMOKE_ROWS = {"train": 120, "validation": 40, "test": 40}

The data files

In [ ]:
# the notebook runs from its own folder, two levels below the experiments folder
PROJECT_DIR = Path.cwd().parents[1]
DATA_DIR = PROJECT_DIR / "data"
CLEANED_PATHS = {
    "train": DATA_DIR / "processed" / data.TRAIN_CLEANED_FILE,
    "validation": DATA_DIR / "processed" / data.VALIDATION_CLEANED_FILE,
    "test": DATA_DIR / "processed" / data.TEST_CLEANED_FILE,
}

The ISG vector folder, outside every checkout

In [ ]:
VECTORS_DIR = Path.home() / "isg_vectors_v2"
if SMOKE:
    VECTORS_DIR = Path.home() / "isg_vectors_v2_smoke" / str(RUN_TAG)
smoke_without_tag = SMOKE and not RUN_TAG
if smoke_without_tag:
    raise ValueError("A smoke run needs a RUN_TAG - it names the smoke vector folder.")

The run name - it is part of every file name of the run

In [ ]:
run_name_parts = [DEVELOPER_INITIALS, "graph", "isg", "embedding", CLASSIFIER]
if SMOKE:
    run_name_parts.append("smoke")
if RUN_TAG:
    run_name_parts.append(str(RUN_TAG))
run_name_parts.append("v2")
RUN_NAME = "_".join(run_name_parts)

A run name that is not a plain file name stops the run

In [ ]:
if not re.fullmatch(r"[\w.-]+", RUN_NAME):
    raise ValueError(
        f"Run name {RUN_NAME!r} is not a plain file name - change RUN_TAG.",
    )

The result folder of the run

In [ ]:
# the result tables are built from results_v2, so a smoke run writes into a folder of its own
RESULTS_DIR = Path.home() / "results_v2"
if SMOKE:
    RESULTS_DIR = Path.home() / "results_v2_smoke" / str(RUN_TAG)
RUN_DIR = RESULTS_DIR / RUN_NAME
EXPERIMENT_NAME = tracking.SMOKE_EXPERIMENT_NAME if SMOKE else tracking.EXPERIMENT_NAME

A missing vector folder stops the run here - the extraction notebook builds it first

In [ ]:
if not VECTORS_DIR.is_dir():
    raise FileNotFoundError(
        f"{VECTORS_DIR} is missing - run isg_vector_extraction.ipynb with the same SMOKE and RUN_TAG first.",
    )

The hash of the manifest of the vector folder - a run or a study of another folder stops

In [ ]:
with (VECTORS_DIR / graph.MANIFEST_FILE).open("rb") as file:
    ISG_MANIFEST_SHA256 = hashlib.file_digest(file, "sha256").hexdigest()

A smoke run reads copies of the first lines of every file - the copies its vector folder was built from

In [ ]:
def copy_first_lines(path: Path, n_lines: int, target_dir: Path) -> Path:
    """Copy the first n_lines of a file into target_dir, under the same file name."""
    with path.open(encoding="utf-8") as file:
        first_lines = list(islice(file, n_lines))
    copy_path = target_dir / path.name
    copy_path.write_text("".join(first_lines), encoding="utf-8")
    return copy_path

In [ ]:
temp_dir = tempfile.TemporaryDirectory()
if SMOKE:
    files_to_read = {}
    for split, path in CLEANED_PATHS.items():
        files_to_read[split] = copy_first_lines(
            path,
            SMOKE_ROWS[split],
            Path(temp_dir.name),
        )
else:
    files_to_read = CLEANED_PATHS

Run context - commit, hashes and versions; refuses uncommitted or unpushed changes

In [ ]:
logging.basicConfig(format="%(asctime)s %(message)s")
run_context = tracking.collect_run_context(PROJECT_DIR, list(files_to_read.values()))
temp_dir.cleanup()
print(f"Run {RUN_NAME}, commit {run_context.environment['commit'][:7]}")
progress_file = progress.ProgressFile(
    Path.home() / "logs" / f"{RUN_NAME}.progress.json",
    RUN_NAME,
)
progress_file.start_stage("loading")

# ISG vectors

The vectors of both texts of every pair of the three splits, as the extraction notebook saved them

In [ ]:
isg_vectors = graph.load_isg_vectors(VECTORS_DIR, run_context)

The pairs of every split

In [ ]:
pairs = {}
for split in features.SPLITS:
    pairs_df = isg_vectors.splits[split].pairs_df
    pairs[split] = pairs_df
    print(f"{split}: {len(pairs_df)} pairs")

The prediction tables are keyed by pair id, so a repeated validation or test id stops the run

In [ ]:
for split in ["validation", "test"]:
    ids_are_unique = pairs[split]["id"].is_unique
    if not ids_are_unique:
        raise ValueError(f"The {split} pairs have repeated ids.")

# Settings

Everything that changes a similarity or the threshold, and the pair counts

In [ ]:
run_params = {
    **RUN_IDENTITY,
    "smoke": SMOKE,
    "mlflow_experiment": EXPERIMENT_NAME,
    "decision_rule": "cosine similarity >= threshold",
    "threshold_candidates": "every validation similarity",
    "isg_manifest_sha256": ISG_MANIFEST_SHA256,
}
if SMOKE:
    for split, n_rows in SMOKE_ROWS.items():
        run_params[f"smoke_rows.{split}"] = n_rows
for split, pairs_df in pairs.items():
    run_params[f"{split}_pairs"] = len(pairs_df)

If the result folder of this run already exists with these params, the run finished before - the cells below then do nothing, as every one starts with if not run_is_complete

In [ ]:
run_is_complete = runs.find_finished_run(RUN_DIR, run_params)
if run_is_complete:
    print(f"{RUN_DIR} exists - the run is already finished.")

# Similarities and the threshold

The cosine similarity of the two texts of every pair of one split, next to its id and label

In [ ]:
def score_split(split: features.SplitMatrices) -> pd.DataFrame:
    """Return the pairs of a split with the cosine similarity of their vectors in the column score.

    Args:
        split: One split of load_isg_vectors.

    Returns:
        One row per pair with the columns id, same and score.
    """
    similarities = features.compute_cosine_similarities((split.first, split.second))
    scores_df = split.pairs_df.copy()
    # the float32 vectors give float32 similarities - float64 keeps their value in the tables
    scores_df["score"] = similarities.astype(np.float64)
    return scores_df

The threshold of the highest F1 among every observed similarity (the lowest one if two are equal),
with the F1 of every candidate

In [ ]:
def select_threshold(scores_df: pd.DataFrame) -> tuple[pd.Series, pd.DataFrame]:
    """Evaluate every observed score as a threshold and return the best row and all rows.

    Args:
        scores_df: Output of score_target_pairs for the validation pairs.

    Returns:
        The row of the best threshold and the table of every candidate threshold.
    """
    candidates = np.unique(scores_df["score"]).tolist()
    thresholds_df = evaluation.evaluate_classification_thresholds(
        scores_df["same"].astype(int),
        scores_df["score"],
        candidates,
    )
    best_row = evaluation.select_best_threshold(thresholds_df)
    return best_row, thresholds_df

Score the validation pairs and select the threshold on them

In [ ]:
if not run_is_complete:
    progress_file.start_stage("validation scoring", n_pairs=len(pairs["validation"]))
    validation_scores_df = score_split(isg_vectors.splits["validation"])
    best_row, thresholds_df = select_threshold(validation_scores_df)
    threshold = float(best_row["threshold"])
    print(f"Threshold {threshold}: validation F1 {best_row['f1']}")

# Evaluate on the test pairs

The test pairs are scored once and never select anything.

In [ ]:
if not run_is_complete:
    progress_file.start_stage("test scoring", n_pairs=len(pairs["test"]))
    started = time.perf_counter()
    test_scores_df = score_split(isg_vectors.splits["test"])
    scoring_seconds_test = time.perf_counter() - started

The prediction tables (one row per pair - id, same, score, pred) and the test metrics

In [ ]:
if not run_is_complete:
    validation_predictions_df = runs.build_predictions(
        pairs["validation"],
        validation_scores_df,
        threshold,
    )
    predictions_df = runs.build_predictions(pairs["test"], test_scores_df, threshold)
    test_metrics = evaluation.evaluate_results(
        predictions_df["same"],
        predictions_df["pred"],
    )
    print(test_metrics)

# Create the confusion matrix

The figure

In [ ]:
if not run_is_complete:
    confusion_figure, axis = plt.subplots(figsize=(6, 5))

The counts of the test pairs

In [ ]:
if not run_is_complete:
    sns.heatmap(
        confusion_matrix(predictions_df["same"], predictions_df["pred"], labels=[0, 1]),
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=[False, True],
        yticklabels=[False, True],
        ax=axis,
        cbar=False,
    )

The labels

In [ ]:
if not run_is_complete:
    axis.set_xlabel("Predicted")
    axis.set_ylabel("Actual")
    axis.set_title(f"Confusion Matrix: {RUN_NAME}")

# Plot the F1 against the threshold

The F1 of every candidate threshold on the validation pairs, the selected threshold dashed

In [ ]:
if not run_is_complete:
    threshold_figure, threshold_axis = plt.subplots(figsize=(9, 5))
    threshold_axis.plot(thresholds_df["threshold"], thresholds_df["f1"], linewidth=2)
    threshold_axis.axvline(threshold, color="grey", linestyle="--")

The labels

In [ ]:
if not run_is_complete:
    threshold_axis.set_xlabel("Threshold")
    threshold_axis.set_ylabel("Validation F1 (%)")
    threshold_axis.set_title(f"F1 vs Threshold: {RUN_NAME}")
    threshold_axis.grid(True)

# Save the results

Everything the run reports is written into one folder; it is uploaded to MLflow from there.

The metrics of the run

In [ ]:
if not run_is_complete:
    run_metrics = {
        "threshold": threshold,
        "validation_f1": float(best_row["f1"]),
        "n_threshold_candidates": len(thresholds_df),
        "scoring_seconds_test": round(scoring_seconds_test, 1),
        "final_n_features": len(isg_vectors.columns),
    }

The test metrics

In [ ]:
if not run_is_complete:
    for name, value in test_metrics.to_dict().items():
        run_metrics[f"test_{name}"] = value

Write the result folder - a half-written .partial folder left by a killed save is removed first

In [ ]:
if not run_is_complete:
    runs.remove_unfinished_run_dir(RUN_DIR)
    tracking.write_run_files(
        RUN_DIR,
        context=run_context,
        params=run_params,
        metrics=run_metrics,
        tables={
            "thresholds-validation": thresholds_df,
            "predictions-validation": validation_predictions_df,
            "predictions-test": predictions_df,
        },
        figures={
            "confusion_matrix-test": confusion_figure,
            "f1_vs_threshold-validation": threshold_figure,
        },
    )

The finished run - the result folder and its metrics

In [ ]:
progress_file.start_stage("done")
print(RUN_DIR)
print((RUN_DIR / tracking.METRICS_FILE).read_text(encoding="utf-8"))